# 🔍 Scout Agent - Job Discovery Engine

**Market Mapper: Real-time Job Discovery using Mistral AI**

This notebook:
- Loads candidate resume JSON and JD JSON
- Uses `mistral-large-latest` with `web_search_preview` tool via **chat.complete** (avoids Agents API rate limits)
- Performs multi-source job searches and outputs a structured match table

In [ ]:
%pip install mistralai python-dotenv --upgrade -q


In [ ]:
import time
import json
from serpapi import GoogleSearch
import os
from dotenv import load_dotenv
from mistralai import Mistral
from pathlib import Path


## 📦 Step 0: Install Dependencies

In [ ]:

try:
    import mistralai
    print(f'✅ mistralai {mistralai.__version__} ready')
except ImportError:
    print('❌ Installation failed. Please restart kernel and try again.')
    raise

Note: you may need to restart the kernel to use updated packages.
✅ mistralai 1.12.4 ready


## 🔑 Step 1: Configure API Key

In [ ]:
load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
if not api_key:
    raise ValueError("❌ MISTRAL_API_KEY not found. Add it to your .env file or set it below.")
    # Uncomment and paste your key here if not using .env:
    # api_key = "your_mistral_api_key_here"

client = Mistral(api_key=api_key)
print('✅ Mistral client initialized')

✅ Mistral client initialized


## 📋 Step 2: Load Resume & JD Data

Place your `resume_analysis.json` in the same folder, or edit the example data below.

In [ ]:
# ── Fallback example data (used if no JSON file is found) ──────────────────────
DEFAULT_RESUME = {
    "personal_info": {"name": "Amir Benali", "location": "Paris, France"},
    "current_title": "Senior Data Scientist",
    "years_of_experience": 7,
    "skills": {
        "programming": ["Python", "SQL", "Scala"],
        "machine_learning": ["XGBoost", "LightGBM", "Scikit-learn"],
        "deep_learning": ["PyTorch", "TensorFlow", "HuggingFace Transformers"],
        "data_engineering": ["Apache Spark", "Airflow", "Kafka", "dbt"],
        "cloud": ["GCP", "BigQuery", "Vertex AI", "AWS"],
        "mlops": ["MLflow", "Docker", "Kubernetes", "Kubeflow"]
    },
    "experience": [
        {
            "title": "Senior Data Scientist",
            "company": "BNP Paribas",
            "years": 3,
            "highlights": [
                "Led team of 4 data scientists on credit risk modeling",
                "Built fraud detection pipeline on GCP processing 500K+ transactions/day",
                "Reduced loan default rate by 18% using XGBoost + SHAP"
            ]
        },
        {
            "title": "Data Scientist",
            "company": "Criteo",
            "years": 3,
            "highlights": [
                "Recommendation engine improving CTR by 23%",
                "NLP pipeline for ad classification using BERT fine-tuning",
                "Reduced model inference time 40% via ONNX quantization"
            ]
        }
    ],
    "certifications": [
        "Google Professional ML Engineer (2023)",
        "AWS Certified ML Specialty (2022)"
    ],
    "strong_match_roles": [
        "Senior Data Scientist",
        "Lead ML Engineer",
        "Applied Scientist",
        "ML Platform Engineer"
    ]
}

DEFAULT_JD = {
    "job_title": "Senior Data Scientist",
    "company": "Google",
    "required_skills": ["Python", "Machine Learning", "Data Analysis", "SQL"],
    "preferred_skills": ["Apache Spark", "GCP", "BigQuery", "Team Leadership"],
    "description": "Develop and deploy large-scale ML models for Google core products. Collaborate cross-functionally, design data pipelines, and drive data-driven decisions at scale."
}

# ── Try loading from file ──────────────────────────────────────────────────────
auditor_resume_json = DEFAULT_RESUME
auditor_jd_json     = DEFAULT_JD
target_company      = DEFAULT_JD["company"]

auditor_result_file = "resume_analysis.json"

if Path(auditor_result_file).exists():
    try:
        with open(auditor_result_file, 'r', encoding='utf-8') as f:
            auditor_data = json.load(f)
        auditor_resume_json = auditor_data.get('resume_analysis', DEFAULT_RESUME)
        auditor_jd_json     = auditor_data.get('jd_analysis',     DEFAULT_JD)
        target_company      = auditor_jd_json.get('company', 'Google')
        print(f"✅ Loaded data from '{auditor_result_file}'")
    except Exception as e:
        print(f"⚠️  Could not load file ({e}). Using built-in example data.")
else:
    print(f"ℹ️  '{auditor_result_file}' not found — using built-in example data.")

print(f"\n📌 Candidate : {auditor_resume_json.get('personal_info', {}).get('name', 'N/A')}")
print(f"📌 Target role: {auditor_jd_json.get('job_title')} @ {target_company}")
print(f"📌 Resume JSON: {len(json.dumps(auditor_resume_json))} chars")
print(f"📌 JD JSON    : {len(json.dumps(auditor_jd_json))} chars")

ℹ️  'resume_analysis.json' not found — using built-in example data.

📌 Candidate : Amir Benali
📌 Target role: Senior Data Scientist @ Google
📌 Resume JSON: 1250 chars
📌 JD JSON    : 392 chars


## 🤖 Step 3: Define Scout System Prompt

In [15]:
SCOUT_SYSTEM_PROMPT = """You are the "Market Mapper" Scout, a high-speed recruitment intelligence agent.
Your purpose is to bridge the gap between a candidate's profile and the real-time 2026 job market.

You have access to the 'web_search' tool. You MUST use it to find current job listings.

Operational Protocol:
1. Analyze the Resume JSON embedded in the user message (not a file — it's inline text).
2. Use web_search to run THREE searches:
   - Search A: Open roles at the target company matching the candidate (e.g. "Google careers senior data scientist 2026")
   - Search B: Similar roles at top competitors (e.g. "senior data scientist jobs Meta Microsoft 2026")
   - Search C: Skill-based discovery using top 3-5 candidate skills (e.g. "PyTorch Spark MLOps jobs 2026")
3. Match each found role against the candidate profile:
   - High match: >80% skill overlap
   - Medium match: 50-80% overlap
   - Strategic Pivot: leverages candidate's niche skills despite lower overall match
4. Output a Markdown table with columns:
   | Job Title | Company | Match Level | Why it Matches | Source Link |
   Include at least 5-8 rows with real URLs from your searches.
5. Stay objective. Use only verified URLs from web searches."""

print('✅ System prompt defined')

✅ System prompt defined


## 🔍 Step 4: Execute Opportunity Scan

> Uses `chat.complete` with `web_search_preview` tool — **avoids the Agents API 429 rate limit**.

In [ ]:
resume_json_str = json.dumps(auditor_resume_json, ensure_ascii=False, indent=2)
jd_json_str     = json.dumps(auditor_jd_json,     ensure_ascii=False, indent=2)

discovery_prompt = f"""I am initiating an Opportunity Scan for the following candidate.

**Target Company:** {target_company}

**Candidate Resume JSON:**
```json
{resume_json_str}
```

**Job Description JSON:**
```json
{jd_json_str}
```

Your Mission:
1. Analyze the resume JSON above and identify the candidate's core competencies.
2. Run your web_search tool for Searches A, B, and C as described in your instructions.
3. Return a Markdown table of 5-8 matched job opportunities with real URLs.

IMPORTANT: You HAVE the web_search tool — use it now. Do not say you cannot search the web."""

print('🚀 Executing Opportunity Scan...')
print('=' * 60)
print(f'📋 Resume JSON : {len(resume_json_str)} chars')
print(f'📋 JD JSON     : {len(jd_json_str)} chars')
print(f'📋 Target      : {target_company}')
print()
print('📡 Sending request to Mistral AI (chat.complete + web_search_preview)...')
print('⏳ This may take 30-60 seconds...')
print()

response = None
MAX_RETRIES = 3

for attempt in range(MAX_RETRIES):
    try:
        response = client.chat.complete(
            model="mistral-small-latest",
            messages=[
                {"role": "system", "content": SCOUT_SYSTEM_PROMPT},
                {"role": "user",   "content": discovery_prompt}
            ],
            # tools=[{"type": "web_search"}]
        )
        print('✅ Response received!')
        break

    except Exception as e:
        error_str = str(e)
        if '429' in error_str and attempt < MAX_RETRIES - 1:
            wait = 30 * (attempt + 1)
            print(f'⏳ Rate limited (429). Waiting {wait}s before retry {attempt+2}/{MAX_RETRIES}...')
            time.sleep(wait)
        else:
            print(f'❌ API Error: {e}')
            print()
            print('💡 Troubleshooting tips:')
            print('   1. Check your MISTRAL_API_KEY is valid and has credits')
            print('   2. Upgrade your Mistral plan for higher web_search quota')
            print('   3. Wait 1-2 minutes and re-run this cell')
            break

🚀 Executing Opportunity Scan...
📋 Resume JSON : 1636 chars
📋 JD JSON     : 444 chars
📋 Target      : Google

📡 Sending request to Mistral AI (chat.complete + web_search_preview)...
⏳ This may take 30-60 seconds...

✅ Response received!


In [ ]:
# ── Search helper ─────────────────────────────────────────────────────────────
def search_jobs_serpapi(query: str, num: int = 5) -> str:
    """Run a Google search via SerpAPI and return formatted results."""
    try:
        results = GoogleSearch({
            "q"       : query,
            "api_key" : SERPAPI_KEY,
            "num"     : num
        }).get_dict()

        lines = []
        for r in results.get("organic_results", []):
            title   = r.get('title', 'No title')
            link    = r.get('link', '#')
            snippet = r.get('snippet', '')[:250]
            lines.append(f"- [{title}]({link})\n  {snippet}")

        if not lines:
            return "No results found."
        return "\n".join(lines)

    except Exception as e:
        return f"Search error: {e}"


# ── Build prompts ─────────────────────────────────────────────────────────────
resume_json_str = json.dumps(auditor_resume_json, ensure_ascii=False, indent=2)
jd_json_str     = json.dumps(auditor_jd_json,     ensure_ascii=False, indent=2)

# Extract top skills from resume for skill-based search
skills = auditor_resume_json.get('skills', {})
top_skills = []
for category in skills.values():
    if isinstance(category, list):
        top_skills.extend(category[:2])
top_skills_str = ' '.join(top_skills[:5])

# ── Run 3 searches ────────────────────────────────────────────────────────────
print('🔍 Running SerpAPI searches...')
print('=' * 60)

query_a = f"{target_company} careers {auditor_jd_json.get('job_title', 'data scientist')} 2026"
query_b = f"{auditor_jd_json.get('job_title', 'senior data scientist')} jobs Meta Microsoft OpenAI 2026"
query_c = f"{top_skills_str} jobs hiring 2026"

print(f'🔎 Search A: {query_a}')
search_a = search_jobs_serpapi(query_a)
print(f'   → {len(search_a)} chars returned')

print(f'🔎 Search B: {query_b}')
search_b = search_jobs_serpapi(query_b)
print(f'   → {len(search_b)} chars returned')

print(f'🔎 Search C: {query_c}')
search_c = search_jobs_serpapi(query_c)
print(f'   → {len(search_c)} chars returned')

print()
print('✅ All searches complete — sending to Mistral for analysis...')
print()

# ── Build final prompt with injected results ──────────────────────────────────
discovery_prompt = f"""I am initiating an Opportunity Scan for the following candidate.

**Target Company:** {target_company}

**Candidate Resume JSON:**
```json
{resume_json_str}
```

**Job Description JSON:**
```json
{jd_json_str}
```

**Live Search Results (fetched via Google):**

🔎 Search A — {target_company} roles:
{search_a}

🔎 Search B — Competitor roles:
{search_b}

🔎 Search C — Skill-based discovery ({top_skills_str}):
{search_c}

**Your Mission:**
1. Analyze the Resume JSON and identify the candidate's core competencies.
2. Cross-reference with the search results above (real, live URLs are already provided).
3. Return a Markdown table of 5-8 best-matched opportunities with these columns:
   | Job Title | Company | Match Level | Why it Matches | Source Link |
   - Match Level: High (>80%), Medium (50-80%), Strategic Pivot (niche match)
   - Why it Matches: reference specific skills/experience from the Resume JSON
   - Source Link: use the URLs from the search results above
"""

# ── Call Mistral (no tools needed — search already done) ──────────────────────
response = None
MAX_RETRIES = 3

for attempt in range(MAX_RETRIES):
    try:
        response = client.chat.complete(
            model="mistral-small-latest",
            messages=[
                {"role": "system", "content": SCOUT_SYSTEM_PROMPT},
                {"role": "user",   "content": discovery_prompt}
            ]
            # ✅ No tools= needed — SerpAPI already fetched the search results
        )
        print('✅ Mistral response received!')
        break

    except Exception as e:
        error_str = str(e)
        if '429' in error_str and attempt < MAX_RETRIES - 1:
            wait = 30 * (attempt + 1)
            print(f'⏳ Rate limited. Waiting {wait}s before retry {attempt+2}/{MAX_RETRIES}...')
            time.sleep(wait)
        else:
            print(f'❌ API Error: {e}')
            break

Note: you may need to restart the kernel to use updated packages.
🔍 Running SerpAPI searches...
🔎 Search A: Google careers Senior Data Scientist 2026
   → 1146 chars returned
🔎 Search B: Senior Data Scientist jobs Meta Microsoft OpenAI 2026
   → 1326 chars returned
🔎 Search C: Python SQL XGBoost LightGBM PyTorch jobs hiring 2026
   → 1219 chars returned

✅ All searches complete — sending to Mistral for analysis...

✅ Mistral response received!


## 📊 Step 5: Display & Save Results

In [31]:
from IPython.display import Markdown, display

if response is None:
    print('⚠️  No response available. Run Step 4 first.')
else:
    print('📊 Scout Agent Results')
    print('=' * 60)

    # Extract text content from response
    content_text = ''

    if hasattr(response, 'choices') and response.choices:
        message = response.choices[0].message

        # Handle list or string content
        if isinstance(message.content, list):
            for block in message.content:
                if hasattr(block, 'text'):
                    content_text += block.text
                elif hasattr(block, 'type') and block.type == 'text':
                    content_text += block.text
        elif isinstance(message.content, str):
            content_text = message.content

        # Log tool calls used
        if hasattr(message, 'tool_calls') and message.tool_calls:
            print(f'🔧 Tools used: {[tc.type if hasattr(tc, "type") else str(tc) for tc in message.tool_calls]}')
            print()

    if content_text:
        display(Markdown(content_text))
    else:
        print('⚠️  Could not extract text content. Raw response:')
        print(response)

    # ── Save to file ──────────────────────────────────────────────────────────
    output_file = f"scout_results_{int(time.time())}.json"
    result_data = {
        "timestamp"       : time.strftime("%Y-%m-%d %H:%M:%S"),
        "target_company"  : target_company,
        "candidate"       : auditor_resume_json.get('personal_info', {}).get('name', 'Unknown'),
        "response_text"   : content_text,
        "response_raw"    : response.model_dump() if hasattr(response, 'model_dump') else str(response)
    }

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(result_data, f, indent=2, default=str)

    print()
    print('=' * 60)
    print(f'✅ Results saved to: {output_file}')

📊 Scout Agent Results


Based on the provided resume JSON and the live search results, here are the top 8 best-matched opportunities for Amir Benali:

| Job Title | Company | Match Level | Why it Matches | Source Link |
|-----------|---------|-------------|----------------|-------------|
| Senior Staff Research Data Scientist, AI Data Intelligence | Google | High | Strong match for Senior Data Scientist role with expertise in machine learning, data analysis, and Python. Preferred skills include Apache Spark and GCP. | [Senior Staff Research Data Scientist, AI Data Intelligence](https://careers.google.com/jobs/results/115626177644634822-senior-staff-research-data-scientist/) |
| Senior Data Scientist, Product, Google Pay | Google | High | Strong match for Senior Data Scientist role with expertise in machine learning, data analysis, and Python. Preferred skills include Apache Spark and GCP. | [Senior Data Scientist, Product, Google Pay](https://careers.google.com/jobs/results/73835978350830278-senior-data-scientist/) |
| Senior Data Scientist - Microsoft Advertising | Microsoft | High | Strong match for Senior Data Scientist role with expertise in machine learning, data analysis, and Python. Experience in developing robust data pipelines and predictive models. | [Senior Data Scientist - Microsoft Advertising](https://www.linkedin.com/jobs/view/senior-data-scientist-microsoft-advertising-at-microsoft-ai-4378756775) |
| Senior Data Scientist | Microsoft AI | High | Strong match for Senior Data Scientist role with expertise in machine learning, data analysis, and Python. Experience in uncovering actionable insights from vast and diverse datasets. | [Senior Data Scientist | Microsoft AI](https://microsoft.ai/job/senior-data-scientist-15/) |
| Lead ML Engineer at Markmonitor | Markmonitor | High | Strong match for Lead ML Engineer role with expertise in Python, SQL, and machine learning frameworks like scikit-learn, XGBoost, LightGBM, and PyTorch. | [Lead ML Engineer at Markmonitor](https://jobgether.com/offer/698e2a700f33c87a10ea9c14-lead-ml-engineer) |
| Principal Data Scientist / ML Architect | Hitachi | High | Strong match for Principal Data Scientist / ML Architect role with expertise in machine learning frameworks such as scikit-learn, TensorFlow, PyTorch, XGBoost, and LightGBM. Proficiency in Python for data science. | [Principal Data Scientist / ML Architect IRC289594 - Careers](https://careers.hitachi.com/jobs/17409996-principal-data-scientist-slash-ml-architect-irc289594) |
| Senior Data Scientist at Microsoft | Microsoft | Medium | Medium match for Senior Data Scientist role with expertise in data-driven insights to drive business growth. Experience with data science and relevant field. | [Senior Data Scientist at Microsoft](https://topjobstoday.com/company/microsoft/job/senior-data-scientist-united-states-washington-redmond-9) |
| Data Scientist, Platform and B2B Products | OpenAI | Medium | Medium match for Data Scientist role with expertise in data science and relevant field. Experience with data-driven insights to drive business growth. | [Careers](https://openai.com/careers/search/) |

These opportunities align well with Amir Benali's skills, experience, and certifications, making them strong candidates for his next career move.


✅ Results saved to: scout_results_1772319411.json
